# Data Connect Python Library - Usage

***Version Note:***
This notebook reflects the *latest version* of the `dataconnect` Python library.
If you are using an older version, the functions or parameters described here may differ from your installed version.

***View version-matched documentation:***
On GitHub, use the branch/tag selector to switch to your version's tag (e.g., `v0.1.0`).

### Setup
Refer to the [Setup](dataconnect_setup.md) and [Quick Start](dataconnect_quickstart.md) documents first.

### Preparation

In [ ]:
# We recommend storing user_token in a separate file or environment variable.
# This is the user authentication token generated from the Developer Center in iMedidata Platform.

user_token = "usertoken"

In [ ]:
# Load prerequisite libraries
from uuid import UUID

import pandas as pd

### Initialize the connection to Data Connect

_Note_: Do remember to close this connection after your operations, or use a context manager (`with` statement) to ensure proper resource management.

In [ ]:
from dataconnect import DataConnectClient

dataconnect_client = DataConnectClient.connect(token=user_token)

### Get all available studies

Get all available studies or search for a study by name.

In [ ]:
try:
    study_result = dataconnect_client.get_studies(search_study_name="")

    if study_result:
        print(f"\n--- Total Studies: {study_result.total_records} ---\n")

        for study in study_result.studies:
            envs_list = ", ".join(env.name for env in study.environments if env.name)
            print(f"• Study: {study.name}")
            if envs_list:
                print(f"  Envs:  {envs_list}\n")
            else:
                print("  Envs:  (none)\n")
    else:
        print("No Studies found.")
except Exception as e:
    print(f"Error: {e}")

### Work with datasets with `dataconnect` library

#### Search datasets in a study environment

Retrieve the study and study environment information from "Developer info" in iMedidata Platform.

In [ ]:
pwb_study = {
    "study_id": "studyid",
    "prod_env_id": "studyenvironmentid",
}

exciter_study = {
    "study_id": "studyid",
    "prod_env_id": "studyenvironmentid",
}

Set search keyword.

In [ ]:
search_name = "datasetname"

Search for a dataset in a specific study environment.

In [ ]:
try:
    datasets = dataconnect_client.get_datasets(
        study_environment_uuid=UUID(exciter_study["prod_env_id"]),
        search_dataset_name=search_name,
        page=1,
        page_size=5,
    )

    print(f"Total datasets available across all pages: {datasets.total_records}")
    print(
        f"Page: {datasets.pagination.page}, "
        f"Page size: {datasets.pagination.page_size}, "
        f"Total pages: {datasets.pagination.total_pages}"
    )

    for dataset in datasets.items:
        print(f"\n  Dataset: {dataset.dataset_name}")
        print(f"  UUID:    {dataset.dataset_uuid}")

except Exception as e:
    print(f"Error: {e}")

#### Identify different versions of a specific dataset

Prepare a list of datasets that are needed.

In [ ]:
# dataset_uuid values can be retrieved from get_datasets()
dataset_ids = {
    "pwb_der_id": "datasetid",
    "pwb_import_id": "datasetid",
    "pwb_rave_id": "datasetid",
    "pwb_custom_id": "datasetid",
    "exciter_der_id": "datasetid",
    "exciter_import_id": "datasetid",
    "exciter_rave_id": "datasetid",
    "exciter_custom_id": "datasetid",
}

Retrieve other dataset versions if they are available.

In [ ]:
try:
    dataset_versions = dataconnect_client.get_dataset_versions(dataset_uuid=UUID(dataset_ids["exciter_custom_id"]))

    for version in dataset_versions:
        print(f"  Dataset: {version.dataset_name}")
        print(f"  Version: {version.dataset_version}")
        print(f"  UUID:    {version.dataset_uuid}\n")

except Exception as e:
    print(f"Error: {e}")

#### Fetch records for specific datasets

In [ ]:
try:
    rave_data = dataconnect_client.fetch_data(dataset_uuid=UUID(dataset_ids["exciter_rave_id"]))
    # Fetching first 3 rows of data
    rave_data.head(3)

except Exception as e:
    print(f"Error: {e}")

In [ ]:
try:
    der_data = dataconnect_client.fetch_data(dataset_uuid=UUID(dataset_ids["pwb_der_id"]))
    # Fetching the first 6 rows of data
    der_data.head()

except Exception as e:
    print(f"Error: {e}")

In [ ]:
try:
    import_data = dataconnect_client.fetch_data(dataset_uuid=UUID(dataset_ids["pwb_import_id"]))
    # Fetching the first 3 rows of data
    import_data.head(3)

except Exception as e:
    print(f"Error: {e}")

In [ ]:
try:
    custom_data = dataconnect_client.fetch_data(dataset_uuid=UUID(dataset_ids["exciter_custom_id"]))
    # Fetching the first 3 rows of data
    custom_data.head(3)

except Exception as e:
    print(f"Error: {e}")

You can also use `first_n_rows` to limit the number of rows fetched, which is useful for large datasets.

In [ ]:
try:
    # Fetch only the first 10 rows from the server
    rave_sample = dataconnect_client.fetch_data(
        dataset_uuid=UUID(dataset_ids["exciter_rave_id"]),
        first_n_rows=10,
    )
    rave_sample.head()

except Exception as e:
    print(f"Error: {e}")

### Data Transformation Example Using pandas

#### Data transformation using native pandas capability in Python

The fetched data is returned as a pandas `DataFrame`. You can use any pandas operations
for data manipulation.

If you have limited memory allocation, we recommend working on a limited set of
records to reduce development time, and then remove the record limit during
recurring execution.

These functions are not provided by the `dataconnect` library. Below are just
examples of how common Python libraries can be used with the `dataconnect` library.
For details on how these libraries can be used, please consult the respective
library documentation.

#### Set up: Using pandas to perform data transformation on fetched DataFrames

In [ ]:
# Pivot der_data
pivot_df = der_data.rename(
    columns={
        "HCT": "HCT_result",
        "PLAT": "plat_result",
        "WBC": "wbc_result",
        "HCT_UNIT": "HCT_unit",
        "PLAT_UNIT": "plat_unit",
        "WBC_UNIT": "wbc_unit",
    }
).melt(
    id_vars=["patient_id", "site_id", "LBTIM"],
    var_name="lab_test_value",
    value_name="value",
)

# Split the lab_test_value column into lab_test and measurement type
pivot_df[["lab_test", "measure"]] = pivot_df["lab_test_value"].str.rsplit("_", n=1, expand=True)
pivot_df = pivot_df.pivot_table(
    index=["patient_id", "site_id", "LBTIM", "lab_test"],
    columns="measure",
    values="value",
    aggfunc="first",
).reset_index()
pivot_df = pivot_df.rename(columns={"result": "lab_result", "unit": "lab_unit"})

# Union the datasets together (pivot_df, import_data)
pivot_df_renamed = pivot_df.rename(
    columns={
        "patient_id": "subjid",
        "site_id": "siteid",
        "lab_test": "lbtest",
        "lab_result": "lbresn",
        "lab_unit": "lbresu",
    }
)
union_df = pd.concat([import_data, pivot_df_renamed], ignore_index=True)

# Join the datasets together (union_df, custom_data) as publish_df
publish_df = import_data.merge(
    union_df[["subjid", "siteid", "visitnum"]].drop_duplicates(),
    on=["subjid", "siteid", "visitnum"],
    how="outer",
)
publish_df.head()

### Publish datasets with `dataconnect` library

#### Set up the project token and publish parameters

The `project_token` is a Base64-encoded token that identifies the target study, study environment, and project.
You can retrieve this from the "Developer info" in the iMedidata Platform.

In [ ]:
project_token = "projecttoken"

# Name for the dataset you want to publish
publish_dataset_name = "my_published_dataset"

# Key columns define the unique key for deduplication
key_columns = ["subjid", "siteid", "visitnum"]

# Source dataset UUIDs that the published dataset is derived from
source_dataset_uuids = [
    UUID(dataset_ids["pwb_import_id"]),
    UUID(dataset_ids["pwb_der_id"]),
]

#### Dry publish (validate without persisting)

Before publishing, use `dry_publish` to validate your dataset against the server
without committing any changes. This lets you catch errors early.

In [ ]:
try:
    dry_result = dataconnect_client.dry_publish(
        project_token=project_token,
        dataset_name=publish_dataset_name,
        key_columns=key_columns,
        source_datasets=source_dataset_uuids,
        data=publish_df,
    )

    print(f"Dry publish status:       {dry_result.status}")
    print(f"Schema valid:             {dry_result.is_schema_valid}")
    print(f"Config valid:             {dry_result.is_config_valid}")
    print(f"Dataset valid:            {dry_result.is_dataset_valid}")
    print(f"Dataset name:             {dry_result.dataset_name}")
    print(f"Dataset version:          {dry_result.dataset_version}")
    print(f"Number of columns:        {dry_result.no_of_columns}")
    print(f"Valid record count:       {dry_result.valid_record_count}")
    print(f"Duplicate record count:   {dry_result.duplicate_record_count}")
    print(f"Invalid record count:     {dry_result.invalid_record_count}")

    if dry_result.errors:
        print(f"\nErrors: {dry_result.errors}")

    if dry_result.invalid_datetime_formats:
        print(f"\nInvalid datetime formats: {dry_result.invalid_datetime_formats}")

    if dry_result.invalid_records is not None and not dry_result.invalid_records.empty:
        print("\nInvalid records:")
        print(dry_result.invalid_records.head())

except Exception as e:
    print(f"Error: {e}")

#### Publish (persist the dataset)

Once the dry publish validates successfully, publish the dataset to the server.

In [ ]:
try:
    publish_result = dataconnect_client.publish(
        project_token=project_token,
        dataset_name=publish_dataset_name,
        key_columns=key_columns,
        source_datasets=source_dataset_uuids,
        data=publish_df,
    )

    print(f"Publish status:           {publish_result.status}")
    print(f"Dataset name:             {publish_result.dataset_name}")
    print(f"Dataset UUID:             {publish_result.dataset_uuid}")
    print(f"Dataset version:          {publish_result.dataset_version}")
    print(f"Dataset batch number:     {publish_result.dataset_batch_number}")
    print(f"Valid record count:       {publish_result.valid_record_count}")
    print(f"Duplicate record count:   {publish_result.duplicate_record_count}")
    print(f"Invalid record count:     {publish_result.invalid_record_count}")

    if publish_result.invalid_records is not None and not publish_result.invalid_records.empty:
        print("\nInvalid records:")
        print(publish_result.invalid_records.head())

except Exception as e:
    print(f"Error: {e}")

You can also specify `datetime_formats` if your dataset contains datetime columns
that need a specific format for validation and publishing.

In [ ]:
try:
    publish_result_with_formats = dataconnect_client.publish(
        project_token=project_token,
        dataset_name=publish_dataset_name,
        key_columns=key_columns,
        source_datasets=source_dataset_uuids,
        data=publish_df,
        datetime_formats={"visit_date": "yyyy-MM-dd"},
    )

    print(f"Publish status: {publish_result_with_formats.status}")

except Exception as e:
    print(f"Error: {e}")

### Close the Client Connection

In [ ]:
if dataconnect_client:
    dataconnect_client.close()